# Clean Up Reddit playlists
attempts to clear empty/small playlists

In [ ]:
PL_DB = {}

Pandas(Index=1, title='x_r.witchHouse_albums_tracks_like', playlistId='PLWptjpDqazOw2MKHCXZWtwpLJubi9Qu8e', thumbnails=[{'url': 'https://lh3.googleusercontent.com/wr28amLh-pMk4vmrYv_Orhly8DTtdvZJFuLwmXG5RNvZJjGlFe_WMnKp4pWlZI1gL7ihQn-xZuzZ0A6VZZbv2Z-iTEH3dpjn=s192', 'width': 192, 'height': 192}, {'url': 'https://lh3.googleusercontent.com/wr28amLh-pMk4vmrYv_Orhly8DTtdvZJFuLwmXG5RNvZJjGlFe_WMnKp4pWlZI1gL7ihQn-xZuzZ0A6VZZbv2Z-iTEH3dpjn=s576', 'width': 576, 'height': 576}], description='Jake G • 0 songs', count='0', author=[{'name': 'Jake G', 'id': 'UCDvJYHQoKPhpF-sGFUMUJqw'}])

In [ ]:
def fetch_playlist(pl_id):
  if pl_id in PL_DB:
    return PL_DB[pl_id]
  else:
    yt_pl = yt.get_playlist(pl_id, limit=1500)
    PL_DB[pl_id] = yt_pl
    return yt_pl


In [ ]:
for p in playlists.itertuples():
  if 'x_r.' not in p.title:
    continue


----------------------------------------------------------------------------------------------------
x_r.witchHouse_albums_tracks_like
move x_r.witchHouse_albums_tracks_like --> x_r.witchHouse_tracks_like
move x_r.witchHouse_albums_tracks_like --> x_r.witchHouse_albums
deleting playlist:  x_r.witchHouse_albums_tracks_like x_r.witchHouse_albums_tracks_like Jake G • 0 songs PLWptjpDqazOw2MKHCXZWtwpLJubi9Qu8e

----------------------------------------------------------------------------------------------------
x_r.triphop_albums_tracks_like
move x_r.triphop_albums_tracks_like --> x_r.triphop_tracks_like
cant find dest x_r.triphop_tracks_like

----------------------------------------------------------------------------------------------------
x_r.treemusic_albums_tracks_like
move x_r.treemusic_albums_tracks_like --> x_r.treemusic_tracks_like
move x_r.treemusic_albums_tracks_like --> x_r.treemusic_albums
deleting playlist:  x_r.treemusic_albums_tracks_like x_r.treemusic_albums_tracks_like J

In [ ]:
REDDIT_PREFIX = 'x_r.'

In [ ]:
# Move tracks_like playlists to tracks_radio if too small
MIN_N_LIKE = 20
delete_pls = []
playlists = pd.DataFrame(yt.get_library_playlists(limit=PLAYLIST_LIMIT))
orig_len = len(playlists)
for p in playlists.itertuples():
  if REDDIT_PREFIX not in p.title: continue
  if '_tracks_like' not in p.title: continue

  track_count = int(p.count.replace(',', ''))
  if track_count < MIN_N_LIKE:
    can_delete = False
    print('\n' + 100*'-' + '\n' + p.title)
    src_vids = frozenset([t['videoId'] for t in fetch_playlist(p.playlistId)['tracks']])
    
    dest_title =  p.title.replace('_tracks_like', '_tracks_radio')
    dest = playlists.loc[playlists.title == dest_title]
    if len(dest):
      dest = dest.iloc[0]
      dest_vids = frozenset([t['videoId'] for t in fetch_playlist(dest.playlistId)['tracks']])
      new_vids = src_vids - dest_vids
      print(f'\ncopy {len(new_vids)} {p.title} --> {dest_title}')
      if len(new_vids):
        res = yt.add_playlist_items(dest.playlistId, videoIds=new_vids)
        if res['status'] == 'STATUS_SUCCEEDED':
          can_delete = True
        else:
          print(f'failed to copy to dest {dest.title}\n{res}')
    else:
      print(f'failed to find dest {dest_title}')


    if  can_delete:
      print('\ndeleting playlist: ', p.title, p.playlistId)
      print(yt.delete_playlist(pl_id))   


----------------------------------------------------------------------------------------------------
x_r.WorldMusic_tracks_like

copy 4 x_r.WorldMusic_tracks_like --> x_r.WorldMusic_tracks_radio
failed to copy to dest x_r.WorldMusic_tracks_radio
{'responseContext': {'visitorData': 'Cgs1SmdRM2U0Nmo0VSiU_6CVBg%3D%3D', 'serviceTrackingParams': [{'service': 'CSI', 'params': [{'key': 'c', 'value': 'WEB_REMIX'}, {'key': 'cver', 'value': '0.1'}, {'key': 'yt_li', 'value': '1'}, {'key': 'EditPlaylist_rid', 'value': '0x4f05a90623c1b65c'}]}, {'service': 'GFEEDBACK', 'params': [{'key': 'logged_in', 'value': '1'}, {'key': 'e', 'value': '24222811,24077266,23744176,24212037,24169501,23918597,24007246,24230438,24135310,24077241,24004644,23998056,23804281,1714258,24211178,23946420,23983296,24187516,24140247,23966208,24034168,24181174,24165080,24167177,23882685,24198739,24230151,24166096,24199724,24215196,24191629,24001373,24187043,23767702,24226208,23934970,24185614,24164186,24211628,23748146,24082661

In [ ]:
# delete reddit playlists with 0 entries
playlists = pd.DataFrame(yt.get_library_playlists(limit=PLAYLIST_LIMIT))
orig_len = len(playlists)
for p in playlists.itertuples():
  if REDDIT_PREFIX not in p.title: continue
  track_count = int(p.count.replace(',', ''))
  if track_count == 0:
    print('deleting playlist: ', p.title, p.title, p.description, p.playlistId)
    print(yt.delete_playlist(p.playlistId), '\n')
playlists = pd.DataFrame(yt.get_library_playlists(limit=PLAYLIST_LIMIT))
print(f'had {orig_len} playlists before, now have {len(playlists)}')


In [ ]:
orig_len = len(playlists)
for p in playlists.itertuples():
  if REDDIT_PREFIX not in p.title: continue

  if '_albums_like' in p.title:
    can_delete = False
    print('\n' + 100*'-' + '\n' + p.title)
    src_vids = frozenset([t['videoId'] for t in fetch_playlist(p.playlistId)['tracks']])


    dest_title =  p.title.replace('_albums_like', '_tracks_like')
    dest = playlists.loc[playlists.title == dest_title]
    if len(dest):
      dest = dest.iloc[0]
      dest_vids = frozenset([t['videoId'] for t in fetch_playlist(dest.playlistId)['tracks']])
      new_vids = src_vids - dest_vids
      print(f'\nmove {len(new_vids)} {p.title} --> {dest_title}')
      if len(new_vids):
        print(yt.add_playlist_items(dest.playlistId, videoIds=new_vids))
      can_delete = True
    else:
      print('cant find dest', dest_title)


    dest_title =  p.title.replace('_albums_like', '_albums')
    dest = playlists.loc[playlists.title == dest_title]
    if len(dest):
      dest = dest.iloc[0]
      dest_vids = frozenset([t['videoId'] for t in fetch_playlist(dest.playlistId)['tracks']])
      new_vids = src_vids - dest_vids
      print(f'\nmove {len(new_vids)} {p.title} --> {dest_title}')
      if len(new_vids):
        print(yt.add_playlist_items(dest.playlistId, videoIds=new_vids))
      can_delete = True
    else:
      print('cant find dest', dest_title)


    # if  can_delete:
    #   print('\ndeleting playlist: ', p.title, p.playlistId)
    #   print(yt.delete_playlist(p.playlistId))    

playlists = pd.DataFrame(yt.get_library_playlists(limit=PLAYLIST_LIMIT))
print(f'had {orig_len} playlists before, now have {len(playlists)}')

  # track_count = int(p.count.replace(',', ''))
  # if track_count < MIN_PL_SIZE:
  #   print('> ', MIN_PL_SIZE, p.title, p.description, p.playlistId)

    # print(pl_vids)
    # if yt_pl['trackCount'] == 0:
    #   print('deleting playlist: ', p.title, p.title, p.description, p.playlistId)
    #   # print(yt.delete_playlist(pplaylistId))


had 525 playlists before, now have 525
